# Credit risk prediction, explainability and bias detection with Amazon SageMaker

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.2.1</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1</strong></li>
</ul>
</div>

### Install packages
Please choose `Python 3 (ipykernel)` kernel to proceed.

We will first install the prerequisite packages. They should be already installed if you run the [Setup.ipynb](../Setup.ipynb) notebook. If you experience problems, uncomment the following three cells to re-install the right libraires and the restart the kernel and then continue

In [ ]:
# --- Lab dependencies (managed via uv) ---------------------------------------
# Installs THIS lab's complete, self-contained kernel dependencies from the
# lab requirements.txt using uv. Idempotent and fast when already satisfied.
# This is the only dependency step the lab needs - Setup.ipynb is not required.
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r requirements.txt

In [ ]:
# # Restart kernel to pick up the updated packages
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

![Credit risk explainability use case](credit_risk_prediction.png)

1. [Overview](#Overview)
1. [Prerequisites and Data](#Prerequisites-and-Data)
    1. [Initialize SageMaker](#Initialize-SageMaker)
    1. [Download data](#Download-data)
    1. [Loading the data: German credit (Update) Dataset](#Loading-the-data:-German-credit-Dataset)
    1. [Data inspection](#Data-inspection)
    1. [Data preprocessing Model and upload to S3](#Preprocess-and-Upload-Training-Data)
1. [Train XGBoost Model](#Train-XGBoost-Model)
1. [Create SageMaker Model with Inference Pipeline](#Create-SageMaker-Model)
1. [Deploy and run inference](#Deploy-Model)
1. [Model Explainability and Bias Detection with SHAP + MLflow (local)](#explainability-mlflow)
    1. [Load the model pipeline locally](#load-pipeline)
    1. [SHAP baseline](#shap-baseline)
    1. [Compute SHAP values](#compute-shap)
    1. [Global explanations](#global-explanations)
    1. [Explain an individual prediction](#individual-explanation)
    1. [Bias detection with standardized metrics](#bias-detection)
    1. [Log the reports to SageMaker managed MLflow](#log-mlflow)
1. [Run the explainability report as a SageMaker managed job](#managed-job)
1. [Clean Up](#Clean-Up)
1. [Additional Resources](#Additional-Resources)

## 1. Overview
Amazon SageMaker helps data scientists and developers to prepare, build, train, and deploy high-quality machine learning (ML) models quickly by bringing together a broad set of capabilities purpose-built for ML.

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>&#9888; Amazon SageMaker Clarify availability change:</strong> New customer access to <strong>Amazon SageMaker Clarify closes on 7/30/26</strong> and no new features are planned. See the <a target="_blank" href="https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-availability-change.html">Clarify availability change</a> guidance. This notebook has been updated to use the AWS-recommended replacement path: the open-source <strong>SHAP</strong> library (the same engine Clarify used internally) for feature attribution, the <strong>standardized bias-metric formulas</strong> computed directly with pandas, and <strong>Amazon SageMaker managed MLflow</strong> for tracking, lineage and governance of the resulting reports.
</div>

In this SageMaker Studio notebook, we highlight how you can use SageMaker to train models, create a deployable SageMaker model, and provide bias detection and explainability to analyze data and understand prediction outcomes from the model.
This sample notebook walks you through:

1. Download and explore credit risk dataset - [South German Credit (UPDATE) Data Set](https://archive.ics.uci.edu/ml/datasets/South+German+Credit+%28UPDATE%29)
2. Preprocessing data with sklearn on the dataset
3. Training GBM model with XGBoost on the dataset
4. Build an inference pipeline model (sklearn model and XGBoost model together) to preprocess input data and produce a prediction outcome per instance
5. Hosting and scoring the single model
6. **Explainability with SHAP** - compute Kernel SHAP feature attributions locally against a mode baseline (global summary + individual prediction waterfall), reproducing what Clarify did on its shadow endpoint.
7. **Bias detection** - compute the standardized pre-training (CI, DPL) and post-training (DPPL, DI) bias metrics directly with pandas.
8. **Log the reports to Amazon SageMaker managed MLflow** for versioning and governance, then re-run the same analysis as a **SageMaker managed processing job**.

![Credit risk explainability model inference](clarify_inf_pipeline_arch.jpg)

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.2.1</strong> and the <strong>SageMaker Python SDK version 3.13.1</strong>.
</div>

## 2. Prerequisites and Data exploration and Feature engineering
### Initialize SageMaker

We use the SageMaker V3 SDK with modular imports from `sagemaker.core` (foundation primitives), `sagemaker.train` (training), and `sagemaker.serve` (inference). Key V3 constructs:

- **`Session`** and **`get_execution_role`** from `sagemaker.core.helper.session_helper`
- **`S3Uploader`** / **`S3Downloader`** from `sagemaker.core.s3` for data transfer
- **`image_uris.retrieve()`** from `sagemaker.core` to get container image URIs

In [ ]:
# cell 01
from io import StringIO
import os
import time
import sys
import IPython
from time import gmtime, strftime

import boto3
import numpy as np
import pandas as pd
import urllib

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.s3 import S3Uploader, S3Downloader
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output,
    ContainerDefinition, InferenceExecutionConfig, ProductionVariant,
)
from sagemaker.core import image_uris
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.utils import repack_model
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import SourceCode, InputData, Compute
from sagemaker.core.shapes import OutputDataConfig

session = Session()
bucket = session.default_bucket()
prefix = "sagemaker/sagemaker-credit-risk-model"
region = session.boto_region_name
role = get_execution_role()

### Download data

First,  __download__ the data and save it in the `data` folder.


$^{[2]}$ Ulrike Grömping
Beuth University of Applied Sciences Berlin
Website with contact information: https://prof.beuth-hochschule.de/groemping/.

In [ ]:
# cell 02
S3Downloader.download(
    "s3://sagemaker-sample-files/datasets/tabular/uci_statlog_german_credit_data/SouthGermanCredit.asc",
    "data",
)

In [ ]:
# cell 03
credit_columns = [
    "status",
    "duration",
    "credit_history",
    "purpose",
    "amount",
    "savings",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "present_residence",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "number_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "credit_risk",
]

`laufkont = status`
                                               
 1 : no checking account                       
 2 : $... < 0$ DM                                
 3 : $0<= ... < 200$ DM                          
 4 : $... >= 200$ DM / salary for at least 1 year

`laufzeit = duration`
     

`moral = credit_history`
                                                
 0 : delay in paying off in the past            
 1 : critical account/other credits elsewhere   
 2 : no credits taken/all credits paid back duly
 3 : existing credits paid back duly till now   
 4 : all credits at this bank paid back duly    

`verw = purpose`
                        
 0 : others             
 1 : car (new)          
 2 : car (used)         
 3 : furniture/equipment
 4 : radio/television   
 5 : domestic appliances
 6 : repairs            
 7 : education          
 8 : vacation           
 9 : retraining         
 10 : business          

`hoehe = amount`
     

`sparkont = savings`
                               
 1 : unknown/no savings account
 2 : $... <  100$ DM             
 3 : $100 <= ... <  500$ DM      
 4 : $500 <= ... < 1000$ DM      
 5 : $... >= 1000$ DM            

`beszeit = employment_duration`
                     
 1 : unemployed      
 2 : $< 1$ yr          
 3 : $1 <= ... < 4$ yrs
 4 : $4 <= ... < 7$ yrs
 5 : $>= 7$ yrs        

`rate = installment_rate`
                   
 1 : $>= 35$         
 2 : $25 <= ... < 35$
 3 : $20 <= ... < 25$
 4 : $< 20$          

`famges = personal_status_sex`
                                         
 1 : male : divorced/separated           
 2 : female : non-single or male : single
 3 : male : married/widowed              
 4 : female : single                     

`buerge = other_debtors`
                 
 1 : none        
 2 : co-applicant
 3 : guarantor   

`wohnzeit = present_residence`
                     
 1 : $< 1$ yr          
 2 : $1 <= ... < 4$ yrs
 3 : $4 <= ... < 7$ yrs
 4 : $>= 7$ yrs        

`verm = property`
                                              
 1 : unknown / no property                    
 2 : car or other                             
 3 : building soc. savings agr./life insurance
 4 : real estate                              

`alter = age`
     
`weitkred = other_installment_plans`
           
 1 : bank  
 2 : stores
 3 : none  

`wohn = housing`
             
 1 : for free
 2 : rent    
 3 : own     

`bishkred = number_credits`
         
 1 : $1$   
 2 : $2-3$ 
 3 : $4-5$ 
 4 : $>= 6$

`beruf = job`
                                               
 1 : unemployed/unskilled - non-resident       
 2 : unskilled - resident                      
 3 : skilled employee/official                 
 4 : manager/self-empl./highly qualif. employee

`pers = people_liable`
              
 1 : 3 or more
 2 : 0 to 2   

`telef = telephone`
                              
 1 : no                       
 2 : yes (under customer name)

`gastarb = foreign_worker`
        
 1 : yes
 2 : no 

`kredit = credit_risk`
         
 0 : bad 
 1 : good


### Data inspection

In [ ]:
# cell 04
training_data = pd.read_csv(
    "data/SouthGermanCredit.asc",
    names=credit_columns,
    header=0,
    sep=r" ",
    engine="python",
    na_values="?",
).dropna()

print(training_data.head())


Plotting histograms for the distribution of the different features is a good way to visualize the data. 


In [ ]:
# cell 05
%matplotlib inline
training_data["credit_risk"].value_counts().sort_values().plot(
    kind="bar", title="Counts of Target", rot=0
)

### Create the raw training and test CSV files

In [ ]:
# cell 06
# prepare raw test data
test_data = training_data.sample(frac=0.1)
test_data = test_data.drop(["credit_risk"], axis=1)
test_filename = "test.csv"
test_columns = [
    "status",
    "duration",
    "credit_history",
    "purpose",
    "amount",
    "savings",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "present_residence",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "number_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
]
test_data.to_csv(test_filename, index=False, header=True, columns=test_columns, sep=",")

# prepare raw training data
train_filename = "train.csv"
training_data.to_csv(train_filename, index=False, header=True, columns=credit_columns, sep=",")

### Encode and Upload Data
Here we encode the training and test data. Encoding input data is not necessary for SageMaker Clarify, but is necessary for XGBoost models.

In [ ]:
# cell 07
test_raw = S3Uploader.upload(test_filename, "s3://{}/{}/data/test".format(bucket, prefix))
print(test_raw)

In [ ]:
# cell 08
train_raw = S3Uploader.upload(train_filename, "s3://{}/{}/data/train".format(bucket, prefix))
print(train_raw)

### Preprocessing and feature engineering with SageMaker Processing job

We use `ScriptProcessor` from `sagemaker.core.processing` to run a sklearn preprocessing script on SageMaker infrastructure. In V3:

- **`ScriptProcessor`** replaces the framework-specific `SKLearnProcessor` — you provide the container image explicitly via `image_uris.retrieve()`
- **`ProcessingInput`** / **`ProcessingS3Input`** and **`ProcessingOutput`** / **`ProcessingS3Output`** are typed shapes from `sagemaker.core.shapes`
- The processing script runs inside the specified sklearn container and produces preprocessed train/val data plus a fitted transformer model

In [ ]:
# cell 09
sklearn_processor = ScriptProcessor(
    image_uri=image_uris.retrieve(
        framework="sklearn", region=region, version="1.4-2",
        py_version="py3", instance_type="ml.m5.large",
    ),
    role=role,
    base_job_name="sagemaker-credit-risk-processing-job",
    instance_type="ml.m5.large",
    instance_count=1,
    sagemaker_session=session,
)

You can have a look at the preprocessing script prepared to run in the processing job

In [ ]:
# cell 10
!pygmentize processing/preprocessor.py

#### NOTE: THIS CELL WILL RUN FOR APPROX. 5-8 MINUTES! PLEASE BE PATIENT. 
For further documentation on SageMaker Processing, you can refer the documentation [here](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html)

In [ ]:
# cell 11
raw_data_path = f"s3://{bucket}/{prefix}/data/train/"
train_data_path = f"s3://{bucket}/{prefix}/data/preprocessed/train/"
val_data_path = f"s3://{bucket}/{prefix}/data/preprocessed/val/"
model_path = f"s3://{bucket}/{prefix}/sklearn/"

sklearn_processor.run(
    code="processing/preprocessor.py",
    inputs=[
        ProcessingInput(
            input_name="raw_data",
            s3_input=ProcessingS3Input(
                s3_uri=raw_data_path,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            )
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train_data",
            s3_output=ProcessingS3Output(
                s3_uri=train_data_path,
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            )
        ),
        ProcessingOutput(
            output_name="val_data",
            s3_output=ProcessingS3Output(
                s3_uri=val_data_path,
                local_path="/opt/ml/processing/val",
                s3_upload_mode="EndOfJob",
            )
        ),
        ProcessingOutput(
            output_name="model",
            s3_output=ProcessingS3Output(
                s3_uri=model_path,
                local_path="/opt/ml/processing/model",
                s3_upload_mode="EndOfJob",
            )
        ),
    ],
    arguments=["--train-test-split-ratio", "0.2"],
    wait=False,
)

In [ ]:
# The processing job above was submitted with wait=False, so .run() returned
# immediately without blocking. Poll the processing job status in a loop here
# until it reaches a terminal state, then fail loudly on any non-Completed outcome.
import time

processing_job = sklearn_processor.latest_job
print(f"Polling processing job: {processing_job.processing_job_name}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    processing_job.refresh()
    status = processing_job.processing_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(
        f"Processing job {processing_job.processing_job_name} ended with status '{status}': "
        f"{processing_job.failure_reason}"
    )
print(f"Processing job {processing_job.processing_job_name} completed successfully.")

## 3. Train XGBoost Model

We train an XGBoost model using the V3 **`ModelTrainer`** class from `sagemaker.train`, replacing the V2 `Estimator` and framework-specific classes.

Key V3 constructs:
- **`ModelTrainer`** — unified training class that works with any container image
- **`SourceCode`** — specifies the training script and source directory
- **`Compute`** — defines instance type and count
- **`OutputDataConfig`** — specifies where to store model artifacts in S3
- **`Channel`** — defines input data channels with explicit `content_type`

In [ ]:
# cell 12
!pygmentize training/train_xgboost.py

### Set up ModelTrainer

We configure the `ModelTrainer` with:
1. **Hyperparameters** — note: `verbosity` replaces the deprecated `silent` parameter in XGBoost 2.0+
2. **Training image** — retrieved via `image_uris.retrieve(framework="xgboost", version="3.0-5")`
3. **Source code** — our custom `train_xgboost.py` training script
4. **Compute** and **OutputDataConfig** — typed configuration objects

In [ ]:
# cell 13
hyperparameters = {
    "max_depth": "5",
    "eta": "0.1",
    "gamma": "4",
    "min_child_weight": "6",
    "verbosity": "0",
    "objective": "binary:logistic",
    "num_round": "100",
    "subsample": "0.8",
    "eval_metric": "auc",
    "early_stopping_rounds": "20",
}

xgboost_image = image_uris.retrieve(
    framework="xgboost", region=region, version="3.0-5",
)

estimator = ModelTrainer(
    training_image=xgboost_image,
    source_code=SourceCode(source_dir="training/", entry_script="train_xgboost.py"),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{bucket}/{prefix}/xgb_model"),
    hyperparameters=hyperparameters,
    compute=Compute(instance_type="ml.r5.xlarge", instance_count=1),
    role=role,
    sagemaker_session=session,
    base_job_name="credit-risk-xgb",
)

### SageMaker Training

We call `estimator.train()` with `Channel` objects that specify `content_type="csv"`. We use `Channel` (from `sagemaker.core.shapes`) instead of `InputData` because `Channel` preserves the `content_type` field required by XGBoost's built-in algorithm container.

#### NOTE: THIS CELL WILL RUN FOR APPROX. 5-8 MINUTES! PLEASE BE PATIENT.

In [ ]:
# cell 14
from sagemaker.core.shapes import Channel, DataSource, S3DataSource

train_channel = Channel(
    channel_name="train", content_type="csv",
    data_source=DataSource(s3_data_source=S3DataSource(
        s3_data_type="S3Prefix", s3_uri=f"s3://{bucket}/{prefix}/data/preprocessed/train/",
        s3_data_distribution_type="FullyReplicated",
    ))
)
val_channel = Channel(
    channel_name="validation", content_type="csv",
    data_source=DataSource(s3_data_source=S3DataSource(
        s3_data_type="S3Prefix", s3_uri=f"s3://{bucket}/{prefix}/data/preprocessed/val/",
        s3_data_distribution_type="FullyReplicated",
    ))
)

training_job = estimator.train(input_data_config=[train_channel, val_channel], wait=False)
TRAINING_JOB_NAME = estimator._latest_training_job.training_job_name

In [ ]:
# The training job was submitted with wait=False, so .train() returned
# immediately. Poll its status until it reaches a terminal state, then fail
# loudly on any non-Completed outcome (mirrors the from-idea-to-production
# reference: a Stopped or Failed job raises, not only Failed).
import time
from sagemaker.core.resources import TrainingJob

training_job = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"Polling training job: {TRAINING_JOB_NAME}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    training_job.refresh()
    status = training_job.training_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status} | secondary={training_job.secondary_status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(
        f"Training job {TRAINING_JOB_NAME} ended with status '{status}': "
        f"{training_job.failure_reason}"
    )
print(f"Training job {TRAINING_JOB_NAME} completed successfully.")

## 4. Create SageMaker Inference Pipeline Model

We prepare a SageMaker inference pipeline using the V3 `Model.create()` API with multiple `ContainerDefinition` objects chained in serial mode.

### Retrieve model artifacts

We access training job artifacts via `training_job.model_artifacts.s3_model_artifacts` (V3 uses dot notation on Pydantic models, not dict subscripting).

In [ ]:
# cell 15
preprocessor_model_data = f"s3://{bucket}/{prefix}/sklearn/model.tar.gz"

training_job = estimator._latest_training_job
job_name = training_job.training_job_name
xgboost_model_data = training_job.refresh().model_artifacts.s3_model_artifacts

print(f"Preprocessor model: {preprocessor_model_data}")
print(f"XGBoost model: {xgboost_model_data}")

### Repack SKLearn Model with Inference Code

In V3, **`repack_model()`** from `sagemaker.core.utils` adds inference code to model artifacts. This replaces V2's framework-specific model objects (`SKLearnModel`) that auto-packaged inference scripts.

The utility downloads the original `model.tar.gz`, adds your inference script to a `code/` subdirectory, and re-uploads.

For hosting this model we provide a custom inference script, that is used to process the inputs and outputs and execute the transform.

The inference script is implemented in the `inference/sklearn/inference.py` file. The custom script defines:

- a custom `input_fn` for pre-processing inference requests. Our input function accepts only CSV input, loads the input in a Pandas dataframe and assigns feature column names to the dataframe
- a custom `predict_fn` for running the transform over the inputs
- a custom `model_fn` for deserializing the model

We will be using the default implementation of the `output_function` provided by SageMaker SKlearn container. To know more, check out: https://github.com/aws/sagemaker-scikit-learn-container



In [ ]:
# cell 16
!pygmentize inference/sklearn/inference.py

Now we repack the sklearn model with the inference script and retrieve the inference container image.

In [ ]:
# cell 17
sklearn_inference_image = image_uris.retrieve(
    framework="sklearn", region=region, version="1.4-2", py_version="py3",
)

# Repack sklearn model with inference code
sklearn_repacked_uri = f"s3://{bucket}/{prefix}/sklearn/repacked/model.tar.gz"
repack_model(
    inference_script="inference.py",
    source_directory="inference/sklearn/",
    dependencies=[],
    model_uri=preprocessor_model_data,
    repacked_model_uri=sklearn_repacked_uri,
    sagemaker_session=session,
)
print(f"Repacked sklearn model: {sklearn_repacked_uri}")

### Repack XGBoost Model with Inference Code

Similarly, we repack the XGBoost model artifacts with its custom inference script.

In [ ]:
# cell 18
!pygmentize inference/xgboost/inference.py

Now we repack the XGBoost model and retrieve the XGBoost 3.0-5 inference image.

In [ ]:
# cell 19
xgboost_inference_image = image_uris.retrieve(
    framework="xgboost", region=region, version="3.0-5",
)

# Repack xgboost model with inference code
xgboost_repacked_uri = f"s3://{bucket}/{prefix}/xgb_model/repacked/model.tar.gz"
repack_model(
    inference_script="inference.py",
    source_directory="inference/xgboost/",
    dependencies=[],
    model_uri=xgboost_model_data,
    repacked_model_uri=xgboost_repacked_uri,
    sagemaker_session=session,
)
print(f"Repacked xgboost model: {xgboost_repacked_uri}")

### Create a SageMaker Inference Pipeline Model

In V3, **`Model.create()`** with a list of **`ContainerDefinition`** objects and **`InferenceExecutionConfig(mode="Serial")`** creates an inference pipeline. This replaces the V2 `PipelineModel` class.

Key parameters:
- **`containers`** — ordered list of `ContainerDefinition` objects; data flows from first to last
- **`InferenceExecutionConfig(mode="Serial")`** — chains containers sequentially
- **`environment`** — each container needs `SAGEMAKER_PROGRAM` and `SAGEMAKER_SUBMIT_DIRECTORY` to locate the inference script

In [ ]:
# cell 20
pipeline_model_name = f"credit-risk-inference-pipeline-{int(time.time())}"

pipeline_model = Model.create(
    model_name=pipeline_model_name,
    containers=[
        ContainerDefinition(
            container_hostname="preprocessing",
            image=sklearn_inference_image,
            model_data_url=sklearn_repacked_uri,
            environment={
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            },
        ),
        ContainerDefinition(
            container_hostname="inference",
            image=xgboost_inference_image,
            model_data_url=xgboost_repacked_uri,
            environment={
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            },
        ),
    ],
    inference_execution_config=InferenceExecutionConfig(mode="Serial"),
    execution_role_arn=role,
)
print(f"Pipeline model: {pipeline_model.model_name}")

### Take note of the `model name` as it will be required while setting up the explainability job.

In [ ]:
# cell 21
pipeline_model.model_name

<a id="Deploy-Model"></a>
## 5. Deploy Model and run inference

V3 deployment uses explicit resource creation:
1. **`EndpointConfig.create()`** with `ProductionVariant` — defines the deployment configuration
2. **`Endpoint.create()`** — creates the endpoint
3. **`endpoint.wait_for_status("InService")`** — waits until ready

This replaces the V2 one-liner `pipeline_model.deploy()`.

#### NOTE: THIS CELL WILL RUN FOR APPROX. 5-8 MINUTES! PLEASE BE PATIENT.

In [ ]:
# cell 22
endpoint_name = f"credit-risk-pipeline-endpoint-{int(time.time())}"
print(f"Endpoint name: {endpoint_name}")

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=pipeline_model_name,
            initial_instance_count=1,
            instance_type="ml.m5.xlarge",
        )
    ],
)

endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_name,
)

endpoint.wait_for_status(target_status="InService")
print(f"Endpoint ready: {endpoint_name}")

### Inference

In V3, **`endpoint.invoke()`** replaces the `Predictor` object pattern. It takes raw `body`, `content_type`, and `accept` parameters and returns an `InvokeEndpointOutput` with a `.body` stream. No serializers/deserializers needed.

In [ ]:
# cell 23
test_dataset = S3Downloader.read_file(test_raw)

response = endpoint.invoke(
    body=test_dataset,
    content_type="text/csv",
    accept="text/csv",
)

result = response.body.read().decode('utf-8')
predictions = [line.split(',') for line in result.strip().split('\n')]
print(f"Got {len(predictions)} predictions")

In [ ]:
# cell 24
predictions

<a id="explainability-mlflow"></a>
## 6. Model Explainability and Bias Detection with SHAP + MLflow

> **Why this changed.** This section previously used **Amazon SageMaker Clarify**. New customer access to Clarify closes on **7/30/26** (see the [Clarify availability change](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-availability-change.html) guidance). AWS recommends reproducing the same analysis with the building blocks Clarify was made of:
>
> * **SHAP** - the exact library Clarify used internally to compute feature attributions.
> * **Standardized bias metrics** (CI, DPL, DPPL, DI) - published arithmetic you compute directly with pandas.
> * **Amazon SageMaker managed MLflow** - to log, version and govern the resulting reports.
>
> In **Part 1** we run the reports **locally** on this notebook kernel. In **Part 2** we run the *same code* as a **SageMaker managed processing job**.

All the reusable logic lives in `explainability/explainability_lib.py`, so the local run and the managed job share one implementation.

In [ ]:
!pygmentize explainability/explainability_lib.py

<a id="load-pipeline"></a>
### 6.1 Load the model pipeline locally

For local explainability we do **not** need a deployed (or shadow) endpoint - we load the trained model artifacts directly and rebuild the same `raw features -> sklearn one-hot transform -> XGBoost -> probability` pipeline that Clarify invoked. We reuse the artifacts produced earlier: the fitted sklearn transformer (`preprocessor_model_data`) and the trained XGBoost booster (`xgboost_model_data`).

In [ ]:
import sys
import importlib

sys.path.insert(0, "explainability")
import explainability_lib as elib
importlib.reload(elib)

from sagemaker.core.s3 import S3Downloader

# download and unpack the two model artifacts locally
os.makedirs("model_artifacts/sklearn", exist_ok=True)
os.makedirs("model_artifacts/xgboost", exist_ok=True)

S3Downloader.download(preprocessor_model_data, "model_artifacts/sklearn")
S3Downloader.download(xgboost_model_data, "model_artifacts/xgboost")

elib.extract_tar("model_artifacts/sklearn/model.tar.gz", "model_artifacts/sklearn")
elib.extract_tar("model_artifacts/xgboost/model.tar.gz", "model_artifacts/xgboost")

# raw-features -> good-credit-probability function (sklearn transform + xgboost)
predict_proba = elib.load_pipeline("model_artifacts/sklearn", "model_artifacts/xgboost")
print("Loaded local raw-features -> probability pipeline")

<a id="shap-baseline"></a>
### 6.2 Create a baseline for SHAP

SHAP values are contrastive: they are computed against a **baseline** sample. We are interested in explaining *bad-credit* predictions, so we want a baseline whose prediction leans toward the *good-credit* class. We use the per-feature [mode](https://en.wikipedia.org/wiki/Mode_(statistics)), a good choice for the mostly-categorical credit features. This is exactly the baseline the Clarify version of this notebook used.

For more on informative vs non-informative baselines, see [SHAP Baselines for Explainability](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-feature-attribute-shap-baselines.html).

In [ ]:
raw_train_df = pd.read_csv("train.csv", header=0, sep=",")

baseline = elib.compute_baseline(raw_train_df)
print("Baseline (mode) row:", baseline)

baseline_proba = float(
    predict_proba(pd.DataFrame([baseline], columns=elib.FEATURE_COLUMNS))[0]
)
print(f"Baseline predicted good-credit probability: {baseline_proba:.4f}")

<a id="compute-shap"></a>
### 6.3 Compute SHAP values locally

We run Kernel SHAP against the mode baseline. We pass `link="logit"` (the equivalent of Clarify's `use_logit=True`) so the expected value and SHAP values are expressed in **log-odds** units and the additive relationship holds:

$$\sum (\text{SHAP values}) + E[y] = \text{model\_prediction\_logit}, \qquad \operatorname{logistic}(z) = \frac{1}{1 + e^{-z}}$$

| Condition | Implication |
|---|---|
| $E[y] > 0$ | baseline probability $> 0.5$ (good-credit baseline) |
| $y < 0$ | predicted probability $< 0.5$ (bad credit) |
| $y > 0$ | predicted probability $> 0.5$ (good credit) |

`num_samples` controls the number of feature coalitions per instance (larger = more accurate, slower). Kernel SHAP is model-agnostic, so this runs on the full sklearn+XGBoost pipeline just like Clarify did.

In [ ]:
test_df = pd.read_csv("test.csv", header=0, sep=",")

explainer, shap_values, expected_value = elib.compute_shap_values(
    predict_proba, baseline, test_df, nsamples=500
)

print("SHAP values shape:", shap_values.shape)
print(f"E[y] (log-odds of the baseline prediction): {expected_value:.4f}")

<a id="global-explanations"></a>
### 6.4 Global explanations

The beeswarm **summary plot** shows, for every feature, how its values push individual predictions toward good (positive SHAP) or bad (negative SHAP) credit. The **bar plot** ranks features by mean absolute SHAP value (global importance). These are the same global views Clarify produced in its explainability report.

In [ ]:
os.makedirs("explainability_report", exist_ok=True)

plot_paths = elib.save_global_plots(shap_values, test_df, "explainability_report", show=True)
plot_paths

<a id="individual-explanation"></a>
### 6.5 Explain an individual bad-credit prediction

We build a per-instance table (prediction + SHAP values + raw features), then draw a **waterfall** plot for the most confident bad-credit prediction. In the plot, $E[f(x)]$ is the good-credit baseline and $f(x)$ is this instance's prediction in log-odds; features with negative SHAP values are the ones driving the decision toward bad credit. Change `min_index` to explain any other instance.

In [ ]:
predictions = predict_proba(test_df)
shap_table = elib.build_shap_table(shap_values, test_df, predictions, expected_value)

# most confident bad-credit prediction = lowest good-credit probability
min_index = int(shap_table["probability_score"].idxmin())
print(f"Explaining instance {min_index} "
      f"(good-credit probability = {shap_table.loc[min_index, 'probability_score']:.4f})")

waterfall_path = elib.save_waterfall(
    shap_values[min_index],
    expected_value,
    test_df[elib.FEATURE_COLUMNS].iloc[min_index].to_numpy(),
    elib.FEATURE_COLUMNS,
    "explainability_report/shap_waterfall_worst_case.png",
    show=True,
)

In [ ]:
# persist the per-instance SHAP table as part of the report
shap_csv = "explainability_report/shap_values.csv"
shap_table.to_csv(shap_csv, index=False)
shap_table.head()

<a id="bias-detection"></a>
### 6.6 Bias detection with standardized metrics

Clarify's bias metrics are **published, standardized formulas** computed from label counts and confusion-matrix values - not a proprietary library. We reproduce a core set directly with pandas (see the [pre-training](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-measure-data-bias.html) and [post-training](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-measure-post-training-bias.html) metric references):

* **CI** - Class Imbalance and **DPL** - Difference in Positive Proportions in true Labels (pre-training)
* **DPPL** - Difference in Positive Proportions in Predicted Labels and **DI** - Disparate Impact (post-training)

We analyze the `age` facet with a threshold of `40` (the convention used by `compute_bias_metrics` is documented in its docstring). Post-training metrics use the model's predictions on the labelled training data.

In [ ]:
import json

train_predictions = predict_proba(raw_train_df)
train_pred_labels = (train_predictions > 0.5).astype(int)

bias_metrics = elib.compute_bias_metrics(
    labels=raw_train_df[elib.LABEL_COLUMN].to_numpy(),
    predictions=train_pred_labels,
    facet_values=raw_train_df["age"].to_numpy(),
    facet_threshold=40,
)

bias_json = "explainability_report/bias_metrics.json"
with open(bias_json, "w") as f:
    json.dump(bias_metrics, f, indent=2)

bias_metrics

<a id="log-mlflow"></a>
### 6.7 Connect to the SageMaker managed MLflow App

We log the reports to an **Amazon SageMaker managed MLflow App** so the explainability and bias results carry the same lineage, versioning and governance as the training runs. The pattern below mirrors `lab-5-mlops/mlflow-tracking/01_mlflow-tracking.ipynb`: we look for an existing MLflow App named `mlflow-app` (created by the workshop setup) and reuse it, otherwise we create one and wait for it to be ready.

First we read the SageMaker Domain / Space / user-profile so the runs are attributed to the current user.

In [ ]:
import json

NOTEBOOK_METADATA_FILE = "/opt/ml/metadata/resource-metadata.json"
domain_id = None
space_name = None

if os.path.exists(NOTEBOOK_METADATA_FILE):
    with open(NOTEBOOK_METADATA_FILE, "rb") as f:
        metadata = json.loads(f.read())
        domain_id = metadata.get("DomainId")
        space_name = metadata.get("SpaceName")

if not space_name:
    raise Exception(
        "Cannot find the current space. Make sure you run this notebook in a "
        "JupyterLab space in Amazon SageMaker Studio."
    )

sm_client = boto3.client("sagemaker")
r = sm_client.describe_space(DomainId=domain_id, SpaceName=space_name)
user_profile_name = r["OwnershipSettings"]["OwnerUserProfileName"]

print(f"SageMaker domain id: {domain_id}")
print(f"Space name: {space_name}")
print(f"User profile name: {user_profile_name}")

In [ ]:
import mlflow
import time
from datetime import date

suffix = date.today().isoformat()
mlflow_name = "mlflow-app"

# Check for an existing MLflow App with the expected name
apps = sm_client.list_mlflow_apps().get("Summaries", [])
mlflow_app = next((a for a in apps if a["Name"] == mlflow_name), None)

if mlflow_app:
    print(f"Using existing MLflow App: {mlflow_app['Name']}")
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app["Arn"])
else:
    print(f"Creating MLflow App: {mlflow_name}...")
    response = sm_client.create_mlflow_app(
        Name=mlflow_name,
        ArtifactStoreUri=f"s3://{bucket}",
        RoleArn=role,
        ModelRegistrationMode="AutoModelRegistrationEnabled",
    )
    while True:
        mlflow_app = sm_client.describe_mlflow_app(Arn=response["Arn"])
        if mlflow_app["Status"] in ["Created", "Updated"]:
            break
        elif mlflow_app["Status"] in ["CreateFailed", "Deleted"]:
            raise RuntimeError(f"MLflow App creation failed: {mlflow_app['Status']}")
        print(f"Status: {mlflow_app['Status']}... waiting")
        time.sleep(30)

while mlflow_app["Status"] in ["Creating", "Updating"]:
    print("MLflow App creating... waiting")
    time.sleep(30)
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app["Arn"])

mlflow_arn = mlflow_app["Arn"]
mlflow_experiment_name = f"sm-id-experiment-{suffix}"

print(f"MLflow App: {mlflow_app['Name']} (v{mlflow_app.get('MlflowVersion', 'N/A')})")
print(f"ARN: {mlflow_arn}")
print(f"Experiment name: {mlflow_experiment_name}")

# point the MLflow client and the LOGNAME/experiment env vars at the managed app
os.environ["MLFLOW_TRACKING_URI"] = mlflow_arn
os.environ["MLFLOW_EXPERIMENT_NAME"] = mlflow_experiment_name
os.environ["LOGNAME"] = user_profile_name
mlflow.set_tracking_uri(mlflow_arn)

### 6.8 Log the explainability + bias report to MLflow

We log the SHAP artifacts (summary plots, waterfall, per-instance SHAP CSV), the per-feature global importance as metrics, the SHAP expected value, and the bias metrics - all under one MLflow run tagged as a `LOCAL` source.

In [ ]:
import math

mlflow.set_experiment(mlflow_experiment_name)

with mlflow.start_run(run_name="explainability-local") as run:
    mlflow.set_tags(
        {
            "mlflow.source.name": "explainable-ai.ipynb",
            "mlflow.source.type": "LOCAL",
            "analysis": "shap-explainability-and-bias",
        }
    )
    mlflow.log_params(
        {
            "num_samples": 500,
            "facet_name": "age",
            "facet_threshold": 40,
            "baseline": ",".join(str(x) for x in baseline),
            "n_explained_instances": int(len(test_df)),
        }
    )

    # global feature importance = mean(|SHAP|) per raw feature
    mean_abs = (
        pd.DataFrame(shap_values, columns=elib.FEATURE_COLUMNS)
        .abs()
        .mean()
        .sort_values(ascending=False)
    )
    for feature, value in mean_abs.items():
        mlflow.log_metric(f"mean_abs_shap_{feature}", float(value))

    mlflow.log_metric("shap_expected_value_logodds", expected_value)
    for name, value in bias_metrics.items():
        if isinstance(value, (int, float)) and math.isfinite(value):
            mlflow.log_metric(f"bias_{name}", float(value))

    mlflow.log_artifacts("explainability_report", artifact_path="explainability_report")
    local_run_id = run.info.run_id

print("Logged local explainability run:", local_run_id)

#### Verify the run was logged under the expected experiment name

We confirm the experiment exists in the managed MLflow App and that our run is the most recent one - the same verification approach used in the lab-5 MLflow notebook.

In [ ]:
experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
assert experiment is not None, f"Experiment '{mlflow_experiment_name}' not found in MLflow"

latest = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=1,
    order_by=["attributes.start_time DESC"],
)
print(f"Experiment '{mlflow_experiment_name}' (id={experiment.experiment_id})")
print("Most recent run id:", latest["run_id"][0])
assert latest["run_id"][0] == local_run_id, "Local run is not the most recent run"
print("Verified: local explainability run is present in the expected experiment.")

In [ ]:
from IPython.display import display, Javascript

presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800,
)["AuthorizedUrl"]

mlflow_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment.experiment_id}/runs/{local_run_id}?workspace=default"
)

# open the MLflow UI, then the specific run
display(Javascript('window.open("{}");'.format(presigned_url)))
display(Javascript('window.open("{}");'.format(mlflow_run_link)))

<a id="managed-job"></a>
## 7. Run the explainability report as a SageMaker managed job

Running locally is great for development, but for a governed, repeatable pipeline you want the analysis to run on **managed infrastructure**. We package the *same* `explainability_lib` code and run `shap_explainability.py` as a **SageMaker Processing job** using `FrameworkProcessor`.

We use `FrameworkProcessor` (rather than `ScriptProcessor`) because its `run()` accepts a `source_dir` and a `requirements.txt`, which it pip-installs into the sklearn container - so `shap`, `mlflow` and the `sagemaker-mlflow` plugin are available inside the job. The job reads the model artifacts and data from S3, recomputes SHAP + bias, and logs its own run to the **same MLflow experiment** (via the `MLFLOW_TRACKING_URI` / `MLFLOW_EXPERIMENT_NAME` / `LOGNAME` environment variables), tagged as a `JOB` source.

In [ ]:
!pygmentize explainability/shap_explainability.py

In [ ]:
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output,
)

explainability_job_output = f"s3://{bucket}/{prefix}/explainability-job"

framework_processor = FrameworkProcessor(
    image_uri=image_uris.retrieve(
        framework="sklearn", region=region, version="1.4-2",
        py_version="py3", instance_type="ml.m5.xlarge",
    ),
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    base_job_name="credit-risk-shap-explainability",
    sagemaker_session=session,
    env={
        "MLFLOW_TRACKING_URI": mlflow_arn,
        "MLFLOW_EXPERIMENT_NAME": mlflow_experiment_name,
        "LOGNAME": user_profile_name,
    },
)

#### NOTE: THIS CELL WILL RUN FOR APPROX. 8-12 MINUTES! PLEASE BE PATIENT.
The job installs the SHAP/MLflow dependencies into the container before running the analysis.

In [ ]:
framework_processor.run(
    code="shap_explainability.py",
    source_dir="explainability",
    requirements="requirements.txt",
    inputs=[
        ProcessingInput(
            input_name="sklearn_model",
            s3_input=ProcessingS3Input(
                s3_uri=preprocessor_model_data,
                local_path="/opt/ml/processing/input/sklearn_model",
                s3_data_type="S3Prefix", s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="xgb_model",
            s3_input=ProcessingS3Input(
                s3_uri=xgboost_model_data,
                local_path="/opt/ml/processing/input/xgb_model",
                s3_data_type="S3Prefix", s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="test",
            s3_input=ProcessingS3Input(
                s3_uri=test_raw,
                local_path="/opt/ml/processing/input/test",
                s3_data_type="S3Prefix", s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="train",
            s3_input=ProcessingS3Input(
                s3_uri=train_raw,
                local_path="/opt/ml/processing/input/train",
                s3_data_type="S3Prefix", s3_input_mode="File",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="report",
            s3_output=ProcessingS3Output(
                s3_uri=explainability_job_output,
                local_path="/opt/ml/processing/output",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
    arguments=["--facet-name", "age", "--facet-threshold", "40", "--num-samples", "500"],
    wait=False,
)

In [ ]:
# The processing job was submitted with wait=False, so run() returned immediately.
# Poll its status until it reaches a terminal state, then fail loudly on any
# non-Completed outcome.
import time

explainability_job = framework_processor.latest_job
print(f"Polling explainability processing job: {explainability_job.processing_job_name}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    explainability_job.refresh()
    status = explainability_job.processing_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(
        f"Explainability job {explainability_job.processing_job_name} ended with "
        f"status '{status}': {explainability_job.failure_reason}"
    )
print(f"Explainability job {explainability_job.processing_job_name} completed successfully.")

#### Verify the managed job logged its run to the expected MLflow experiment

The job logs a run tagged with `mlflow.source.type = 'JOB'`. We search the same experiment for that run to confirm the managed job tracked its report to MLflow.

In [ ]:
experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
assert experiment is not None, f"Experiment '{mlflow_experiment_name}' not found in MLflow"

job_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.`mlflow.source.type` = 'JOB'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)

assert len(job_runs) > 0, "No JOB-sourced run found in the experiment yet"
managed_run_id = job_runs["run_id"][0]
print("Managed-job run id:", managed_run_id)
job_runs[[c for c in ["run_id", "tags.mlflow.runName", "status"] if c in job_runs.columns]]

In [ ]:
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800,
)["AuthorizedUrl"]

mlflow_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment.experiment_id}/runs/{managed_run_id}?workspace=default"
)
display(Javascript('window.open("{}");'.format(mlflow_run_link)))

<a id="Clean-Up"></a>
## 8. Clean Up

We delete the inference endpoint and related resources created for the inference demo. The SHAP explainability ran locally and as a processing job, so there is no separate explainability endpoint to tear down (unlike the Clarify shadow endpoint). The MLflow App is a shared, managed resource and is intentionally **not** deleted here.

In [ ]:
endpoint.delete()
endpoint_config.delete()
pipeline_model.delete()
print("All resources deleted")

<a id="Additional-Resources"></a>
## 9. Additional Resources to explore

* [Amazon SageMaker Clarify availability change (migration guidance)](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-availability-change.html)
* [SHAP documentation](https://shap.readthedocs.io/en/latest/)
* [Clarify pre-training bias metrics reference](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-measure-data-bias.html)
* [Clarify post-training bias metrics reference](https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-measure-post-training-bias.html)
* [Amazon SageMaker managed MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
* [AWS-published SageMaker AI monitoring reference solutions](https://github.com/aws-samples/sample-aiops-on-amazon-sagemakerai/tree/main/monitoring)
* [Fairness Measures for Machine Learning in Finance](https://pages.awscloud.com/rs/112-TZM-766/images/Fairness.Measures.for.Machine.Learning.in.Finance.pdf)